# The Keras functional API
Three small non-linear models (two outputs, two inputs, a skip connection), then a real one: VGG16's frozen
convolutional base with two heads that predict a person's age and gender from a face photo (UTKFace),
compared with two separate single-output models (3 seeds each).

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow's start-up messages
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"  # take GPU memory only as needed
os.environ["TF_CUDNN_WORKSPACE_LIMIT_IN_MB"] = "256"   # the GPU is shared; keep scratch memory small
import warnings
warnings.filterwarnings("ignore", category=UserWarning)   # Keras notices about tf.data
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers

tf.get_logger().setLevel("ERROR")
HERE = Path.cwd().parent if Path.cwd().name == ".logs" else Path.cwd()   # the Note folder, also from .logs/
(HERE / "data").mkdir(exist_ok=True)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))

## 1. Two outputs from one input

In [ ]:
from keras import Model
from keras.layers import Input, Dense, Concatenate, Add, Conv2D

x = Input(shape=(3,), name="input")                        # salary, height, marital status
hidden1 = Dense(128, activation="relu", name="hidden1")(x)
hidden2 = Dense(64, activation="relu", name="hidden2")(hidden1)
output1 = Dense(1, activation="linear", name="age")(hidden2)     # regression
output2 = Dense(1, activation="sigmoid", name="place")(hidden2)  # Delhi (0) or Mumbai (1)
model = Model(inputs=x, outputs=[output1, output2])
model.summary()

## 2. Two inputs, one output

In [ ]:
input_a = Input(shape=(32,), name="input_a")
input_b = Input(shape=(128,), name="input_b")
a = Dense(8, activation="relu")(input_a)
a = Dense(4, activation="relu")(a)
b = Dense(64, activation="relu")(input_b)
b = Dense(32, activation="relu")(b)
b = Dense(4, activation="relu")(b)
combined = Concatenate(name="combined")([a, b])            # 4 + 4 = 8 numbers
z = Dense(2, activation="relu")(combined)
z = Dense(1, activation="linear", name="output")(z)
model2 = Model(inputs=[input_a, input_b], outputs=z)
model2.summary()
print(model2.predict([np.ones((1, 32)), np.ones((1, 128))], verbose=0).shape)

## 3. A skip connection

In [ ]:
inputs = Input(shape=(32, 32, 16))
y = Conv2D(16, 3, padding="same", activation="relu")(inputs)
y = Conv2D(16, 3, padding="same")(y)
outputs = Add(name="skip")([inputs, y])                    # the input jumps over the two convolutions
block = Model(inputs, outputs)
block.summary()

## 4. Age and gender from one face photo (UTKFace)

In [ ]:
# UTKFace (Zhang, Song and Qi 2017): 23,708 aligned and cropped face photos, 200 x 200 pixels.
# Each file name starts with age_gender_race, gender 0 = male, 1 = female. Non-commercial research use only.
FACES = Path.home() / "datasets" / "UTKFace"
if not FACES.exists():
    tar = keras.utils.get_file("UTKFace.tar.gz",
          "https://huggingface.co/datasets/py97/UTKFace-Cropped/resolve/main/UTKFace.tar.gz",
          cache_dir=Path.home(), cache_subdir="datasets")
    import shutil
    shutil.unpack_archive(tar, FACES.parent)
records = []
for p in sorted(FACES.glob("*.jpg")):
    parts = p.name.split("_")
    if len(parts) >= 4 and parts[1] in ("0", "1"):         # a few names lack a field; skip them
        records.append(dict(file=p, age=int(parts[0]), gender=int(parts[1])))
faces = pd.DataFrame(records)
print(len(faces), "photos | age", faces.age.min(), "to", faces.age.max(), "median", faces.age.median(),
      "| female share", round(faces.gender.mean(), 3))

In [ ]:
# 80% training, 20% test, fixed once.
rng = np.random.default_rng(0)
order = rng.permutation(len(faces))
train_df, test_df = faces.iloc[order[:int(0.8 * len(faces))]], faces.iloc[order[int(0.8 * len(faces)):]]
print(len(train_df), "training photos,", len(test_df), "test photos")

In [ ]:
from keras.applications.vgg16 import VGG16, preprocess_input
SIZE = 128
conv_base = VGG16(weights="imagenet", include_top=False, input_shape=(SIZE, SIZE, 3))
conv_base.trainable = False

# The full model, built with the functional API: one input, a shared frozen base, two heads.
image = conv_base.input
flat = layers.Flatten(name="flatten")(conv_base.output)
age_hidden = Dense(256, activation="relu", name="age_hidden")(flat)
gender_hidden = Dense(256, activation="relu", name="gender_hidden")(flat)
age = Dense(1, activation="linear", name="age")(age_hidden)
gender = Dense(1, activation="sigmoid", name="gender")(gender_hidden)
full = Model(inputs=image, outputs=[age, gender], name="age_gender")
print(f"total {full.count_params():,} parameters, trainable "
      f"{sum(int(np.prod(w.shape)) for w in full.trainable_weights):,}")
print([l.name for l in full.layers][-6:])

In [ ]:
# The base is frozen and the photos are not augmented, so the base turns each photo into the same numbers in every
# epoch. We therefore run every photo through the base once and train only the heads on those numbers. This gives
# the same model as training `full`, much faster.
def features(df):
    out = []
    for start in range(0, len(df), 256):
        batch = np.stack([keras.utils.img_to_array(keras.utils.load_img(f, target_size=(SIZE, SIZE)))
                          for f in df.file.iloc[start:start + 256]])
        out.append(conv_base.predict(preprocess_input(batch), verbose=0))
    return np.concatenate(out)

F_train, F_test = features(train_df), features(test_df)
print("features per photo:", F_train.shape[1:])
y_train = {"age": train_df.age.values.astype("float32"), "gender": train_df.gender.values.astype("float32")}
y_test = {"age": test_df.age.values.astype("float32"), "gender": test_df.gender.values.astype("float32")}

In [ ]:
def heads(outputs):
    """The part of the model above the frozen base, for 'age', 'gender' or both."""
    f = Input(shape=F_train.shape[1:], name="base_output")
    flat = layers.Flatten()(f)
    outs = []
    if "age" in outputs:
        outs.append(Dense(1, name="age")(Dense(256, activation="relu")(flat)))
    if "gender" in outputs:
        outs.append(Dense(1, activation="sigmoid", name="gender")(Dense(256, activation="relu")(flat)))
    loss = {"age": "mae", "gender": "binary_crossentropy"}
    metric = {"age": ["mae"], "gender": ["accuracy"]}
    if len(outs) == 2:                                     # two outputs: one loss per output, added with weights
        model = Model(f, outs)
        model.compile(optimizer="adam", loss=loss, loss_weights={"age": 0.1, "gender": 1.0}, metrics=metric)
    else:
        model = Model(f, outs[0])
        model.compile(optimizer="adam", loss=loss[outputs[0]], metrics=metric[outputs[0]])
    return model

EPOCHS = 15
results, curves = [], []
for seed in (0, 1, 2):
    for name, outputs in (("two_outputs", ("age", "gender")), ("age_only", ("age",)), ("gender_only", ("gender",))):
        keras.utils.set_random_seed(seed)
        m = heads(outputs)
        ytr = {k: y_train[k] for k in outputs} if len(outputs) == 2 else y_train[outputs[0]]
        yte = {k: y_test[k] for k in outputs} if len(outputs) == 2 else y_test[outputs[0]]
        h = m.fit(F_train, ytr, epochs=EPOCHS, batch_size=64, validation_data=(F_test, yte), verbose=0)
        pred = m.predict(F_test, verbose=0)
        pred = pred if isinstance(pred, (list, tuple)) else [pred]
        row = dict(model=name, seed=seed, head_params=m.count_params())
        for k, p in zip(outputs, pred):
            p = np.asarray(p).ravel()
            row["age_mae" if k == "age" else "gender_accuracy"] = float(np.abs(p - y_test["age"]).mean() if k == "age"
                                                                         else ((p > 0.5) == y_test["gender"]).mean())
        results.append(row)
        if name == "two_outputs":
            for e in range(EPOCHS):
                curves.append(dict(seed=seed, epoch=e + 1, val_age_mae=h.history["val_age_mae"][e],
                                   val_gender_accuracy=h.history["val_gender_accuracy"][e]))
        print(row)
results = pd.DataFrame(results).round(4)
results.to_csv(HERE / "data" / "results.csv", index=False)
pd.DataFrame(curves).round(4).to_csv(HERE / "data" / "two_output_curves.csv", index=False)

In [ ]:
print(results.groupby("model")[["head_params", "age_mae", "gender_accuracy"]].mean().round(3))
baseline_mae = np.abs(y_test["age"] - np.median(y_train["age"])).mean()
majority = max(y_test["gender"].mean(), 1 - y_test["gender"].mean())
print(f"guessing the median training age for everyone: MAE {baseline_mae:.2f} years;"
      f" guessing the commoner gender: accuracy {majority:.3f}")